In [0]:
dbutils.widgets.text("folder_path", "/mnt/landing/", "enter folder path")
folder_path = dbutils.widgets.get("folder_path")
print("watching folder -->", folder_path)

In [0]:
folder_path = "abfss://bigdata-5@bigdatastoragee.dfs.core.windows.net/Landing "
files = dbutils.fs.ls(folder_path)
latest_file = max(files, key=lambda f: f.modificationTime)

file_name = latest_file.name
file_location = latest_file.path

print("new file name -->", file_name)
print("new file location -->", file_location)

In [0]:
dbutils.widgets.text("folder_path", "/mnt/landing/", "enter folder path")

folder_path = dbutils.widgets.get("folder_path")

print("watching folder -->", folder_path)
folder_path = "abfss://bigdata-5@bigdatastoragee.dfs.core.windows.net/Landing "

files = dbutils.fs.ls(folder_path)

In [0]:
latest_file = max(files, key=lambda f: f.modificationTime)

file_name = latest_file.name
file_location = latest_file.path

print("new file name -->", file_name)
print("new file location -->", file_location)

In [0]:
dbutils.widgets.text("folder_path", "/mnt/landing/", "enter folder path")

folder_path = dbutils.widgets.get("folder_path")

print("watching folder -->", folder_path)

folder_path = "abfss://bigdata-5@bigdatastoragee.dfs.core.windows.net/Landing "

files = dbutils.fs.ls(folder_path)

print("Files in folder:")
for file in files:
    print(file.name)

# Read all CSV files from the folder
df = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .csv(folder_path)

display(df)

In [0]:
from pyspark.sql.functions import col, current_timestamp

df2 = (
    spark.read
    .option("header", True)
    .csv("abfss://bigdata-5@bigdatastoragee.dfs.core.windows.net/Landing ")
    .withColumn("Ingest_Timestamp", current_timestamp())
    .withColumn("ingested_filename", col("_metadata.file_name"))
)
display(df2)

In [0]:
df1 = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv("abfss://bigdata-5@bigdatastoragee.dfs.core.windows.net/Landing ")
    .withColumn("Ingest_Timestamp", current_timestamp())
    .withColumn("ingested_filename", col("_metadata.file_name"))
)

joined_df = df1.join(
    df2,
    (df1["Origin"] == df2["Origin"]) &
    (df1["Dest"] == df2["Dest"]),
    "left"
)

In [0]:
display(joined_df)

In [0]:
from pyspark.sql import functions as F

folder_path = "abfss://bigdata-5@bigdatastoragee.dfs.core.windows.net/Landing "
df = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .csv(folder_path)

df = df.withColumn(
    "departure_delay_data_quality",
    F.when( F.col("DepDelay").cast("double").between(-120, 1440),1).otherwise(0)
)

df.select(
    "DepDelay",
    "departure_delay_data_quality"
).show()

In [0]:
from pyspark.sql import functions as F

df = df.withColumn(
    "distance_data_quality",
    F.when(F.col("Distance").cast("double") > 0,1).otherwise(0)
)

df.select(
    "Distance",
    "distance_data_quality"
).show()

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.functions import col

data = [
    (1, "A", None),
    (2, None, "X"),
    (3, "C", None),
    (4, "D", "Y"),
    (5, None, None),
    (6, "F", None)
]

df_test = spark.createDataFrame(data, ["col1", "col2", "col3"])

# df_test.show()

# df_test.select(
#     F.round(
#         F.avg(col("col2").isNull().cast("int")) * 100, 2
#     ).alias("col2")
# ).show()

mylist = [
    F.round(
        F.avg(col("col2").isNull().cast("int")) * 100, 2
    ).alias("col2"),

    F.round(
        F.avg(col("col3").isNull().cast("int")) * 100, 2
    ).alias("col3")
]

# print(mylist, len(mylist))

df_test.select(mylist).show()

# df_test.select(col("col2").isNull()).show()

In [0]:
df_bronze = df.withColumns({
    # Presence checks (1 = present, 0 = missing)
    "null_flight_date_dq": F.col("Year").isNull().cast("int"),
    "null_airline_dq": F.col("Reporting_Airline").isNull().cast("int"),
    "null_origin_dq": F.col("Origin").isNull().cast("int"),
    "null_dest_dq": F.col("OriginState").isNull().cast("int")
})

display(df_bronze)

In [0]:
df_bronze = df_bronze.withColumns({
    "dq_depdelay_range":
        (~F.col("DepDelay").cast("double").between(-120, 1440)).cast("int"),

    "dq_distance_range":
        (~F.col("Distance").cast("double").between(1, 10000)).cast("int")
})

display(df_bronze)

In [0]:
df_bronze = df_bronze.withColumns({
    "delay_range_dq":
        (~F.col("DepDelay").cast("double").between(-120, 1440)).cast("int"),

    "distance_range_dq":
        (~F.col("Distance").cast("double").between(1, 10000)).cast("int")
})

df_bronze.show()

In [0]:
df_bronze = df_bronze.fillna(0, subset=["delay_range_dq", "distance_range_dq"])
display(df_bronze)

In [0]:
df_cols = [c for c in df_bronze.columns if c.endswith("_dq")]
print(df_cols)
df_bronze = df_bronze.withColumn(
    "total_flag", sum(F.coalesce(F.col(c), F.lit(0)) for c in df_cols))\
        .withColumn("dg_is_clean", (F.col("total_flag") == 0).cast("int"))

display(df_bronze)

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.functions import col

dg_cols = [c for c in df_bronze.columns if c.endswith("_dg")]
print(dg_cols)

df_bronze = df_bronze.withColumn(
    "dg_total_flags", sum((F.coalesce(F.col(c).cast("int"), F.lit(0)) for c in dg_cols), F.lit(0)))\
        .withColumn("dg_is_clean", (F.col("dg_total_flags") == 0).cast("int"))

display(df_bronze)

In [0]:
total_rows = df_bronze.count()
clean_pct = df_bronze.filter(F.col("dg_is_clean") == 1).count() / total_rows * 100
print(f"Clean data percentage: {clean_pct:.2f}%")

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.functions import col
from pyspark.sql.types import StructField , StructType , StringType , DoubleType , IntegerType

df_airport = (
    spark.read.option("multiline","true").json("abfss://bigdata-5@bigdatastoragee.dfs.core.windows.net/Landing ")
    .filter(F.col("latitude") .cast("double").between(-90,90))
    .filter(F.col("longitude") .cast("double").between(-180,180))
    .withColumn("_load_ts_",F.current_timestamp())
)
# df_airport.count()
display(df_airport)

In [0]:
from pyspark.sql.functions import round, col

df = df.withColumn("latitude", round(col("latitude").cast("double"), 2)) \
       .withColumn("longitude", round(col("longitude").cast("double"), 2))

In [0]:
df_airport.select(
    round(col("latitude").cast("double"), 2).alias("latitude"),
    round(col("longitude").cast("double"), 2).alias("longitude")
)
display(df_airport)

In [0]:
from pyspark.sql.functions import when, col

df_corrput = df.withColumn(
    "corrput_record",
    when(col("corrput_record").isNull(), 0)
    .when(col("corrput_record") == 0, 1)
    .otherwise(col("corrput_record"))
)

In [0]:
airport_schema = StructType([
    StructField("altitude", IntegerType(), True),
    StructField("city", StringType(), True),
    StructField("country", StringType(), True),
    StructField("iata", StringType(), True),
    StructField("icao", StringType(), True),
    StructField("latitude", DoubleType(), True),
    StructField("longitude", DoubleType(), True),
    StructField("alt", DoubleType(), True),
    StructField("tz_offset", DoubleType(), True),
    StructField("dst", StringType(), True),
    StructField("timezone", StringType(), True),
    StructField("type", StringType(), True),
    StructField("source", StringType(), True)
])

In [0]:
# Write data to Silver Delta table
df_airport.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("default.silver_airport")

In [0]:
display(spark.table("default.silver_airport"))

In [0]:

# Save Bronze-processed airport data in Delta format

delta_path = "abfss://bigdata-5@bigdatastoragee.dfs.core.windows.net/Silver/airport_delta"

df_airport.write \
    .format("delta") \
    .mode("overwrite") \
    .save(delta_path)

print("Airport data saved in Delta format!")


In [0]:
from pyspark.sql import functions as F

base_path = "abfss://bigdata-5@bigdatastoragee.dfs.core.windows.net/"

# 1. Read flight CSV
flights_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .option("mode", "PERMISSIVE")
    .csv(base_path + "Landing /On_Time_Reporting_Carrier_On_Time_Performance_(1987_present)_2022_4.csv")
)

# 2. Read airport JSON
airports_df = (
    spark.read
    .option("multiline", True)
    .option("inferSchema", True)
    .json(base_path + "Landing /airports.json")
)

# 3. Prepare airport columns for origin join
origin_airports = airports_df.select(
    F.col("iata").alias("Origin"),
    F.col("name").alias("OriginAirportName"),
    F.col("city").alias("OriginAirportCity"),
    F.col("country").alias("OriginAirportCountry"),
    F.col("latitude").cast("double").alias("OriginLatitude"),
    F.col("longitude").cast("double").alias("OriginLongitude"),
    F.col("altitude").cast("int").alias("OriginAltitude"),
    F.col("timezone").alias("OriginTimezone")
)

# 4. Prepare airport columns for destination join
destination_airports = airports_df.select(
    F.col("iata").alias("Dest"),
    F.col("name").alias("DestAirportName"),
    F.col("city").alias("DestAirportCity"),
    F.col("country").alias("DestAirportCountry"),
    F.col("latitude").cast("double").alias("DestLatitude"),
    F.col("longitude").cast("double").alias("DestLongitude"),
    F.col("altitude").cast("int").alias("DestAltitude"),
    F.col("timezone").alias("DestTimezone")
)

# 5. Join flight data with origin and destination airports
joined_df = (
    flights_df
    .join(origin_airports, on="Origin", how="left")
    .join(destination_airports, on="Dest", how="left")
)

# Convert flight times into minutes after midnight
joined_df = (
    joined_df
    .withColumn("schedule_crsdeparture_minutes",
                F.floor(F.col("CRSDepTime").cast("int") / 100) * 60 + F.col("CRSDepTime").cast("int") % 100)
    .withColumn("schedule_departure_minutes",
                F.floor(F.col("DepTime").cast("int") / 100) * 60 + F.col("DepTime").cast("int") % 100)
    .withColumn("schedule_crsarrival_minutes",
                F.floor(F.col("CRSArrTime").cast("int") / 100) * 60 + F.col("CRSArrTime").cast("int") % 100)
    .withColumn("schedule_arrival_minutes",
                F.floor(F.col("ArrTime").cast("int") / 100) * 60 + F.col("ArrTime").cast("int") % 100)
)

# 7. Save the joined data as Delta
delta_path = base_path + "Silver/flight_airport_delta"

(
    joined_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(delta_path)
)

# 8. Verify the saved Delta table
result_df = spark.read.format("delta").load(delta_path)

display(result_df)